# AI for Meter Reading — Pipeline complet

**Objectif :** à partir d'une simple photo d'un compteur d'eau, lire automatiquement les trois chiffres noirs (partie entière, en m³) affichés sur le cadran.

Ce notebook reconstruit proprement le pipeline développé pendant le projet, tel que décrit dans le rapport et le README :

1. **Détection et découpage du cadran** — YOLOv8 (modèle entraîné en 2 passes, sur des labels Roboflow)
2. **Alignement du cadran** — détection de deux repères (boîte noire / boîte rouge) puis rotation automatique
3. **Suppression de la partie rouge** — la partie rouge du cadran (décimales) n'est pas informative pour le nombre de m³
4. **Lecture des chiffres** — solution retenue : **EasyOCR** (les tentatives de CNN maison et de ResNet fine-tuné sont documentées en annexe, avec leurs résultats, mais n'ont pas été retenues)

**Dataset :** 795 images de compteurs d'eau, souvent floues, mal orientées ou prises de loin. Après la première étape de détection, 726 des 795 images sont correctement recadrées.

**Résultat final (rapport) :** score de soumission ≈ **0.284** avec EasyOCR sur les images alignées, contre une accuracy exacte de seulement **0.68 %** pour le CNN maison entraîné from scratch.

> ⚠️ Ce notebook a été reconstruit à partir de deux notebooks de travail (dont certaines cellules étaient tronquées, notamment la boucle d'alignement et les boucles d'entraînement des CNN) et du rapport/README du projet. Les chemins ci-dessous sont à adapter à votre arborescence — voir la cellule de configuration. Les fichiers de données, labels et poids YOLO (`.pt`) ne sont pas inclus : c'est à vous de les remettre aux bons chemins (ou de les redemander à ta copine !).


## 0. Configuration & installation

Tout se pilote depuis le dictionnaire `CONFIG` ci-dessous : changez `BASE_DIR` pour pointer vers votre dossier de travail (localement ou sur Google Drive) et le reste des chemins suit automatiquement.


In [ ]:
# Décommentez si vous exécutez ce notebook sur Google Colab avec les données sur Drive
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
!pip install -q ultralytics easyocr opencv-python-headless pandas tqdm matplotlib pyyaml torch torchvision


In [ ]:
import os
import re
import math
import shutil

import cv2
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

from ultralytics import YOLO
import easyocr

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models


In [ ]:
# === CONFIGURATION ==========================================================
# Adaptez BASE_DIR à votre projet (dossier local ou chemin Google Drive)
BASE_DIR = "/content/drive/My Drive/Cadrants"   # <-- à adapter

CONFIG = {
    # images brutes du challenge (les 795 photos de compteurs)
    "raw_images_dir": os.path.join(BASE_DIR, "pictures"),

    # --- Etape 1 : detection / decoupage du cadran ---
    "yolo_crop_yaml": os.path.join(BASE_DIR, "cadrants.v4i.yolov8", "data.yaml"),
    "yolo_crop_train_images": os.path.join(BASE_DIR, "cadrants.v4i.yolov8", "train", "images"),
    "yolo_crop_weights": os.path.join(BASE_DIR, "modele", "best.pt"),
    "crops_dir": os.path.join(BASE_DIR, "crops"),

    # --- Etape 2 : detection des reperes noir / rouge + alignement ---
    "yolo_align_yaml": os.path.join(BASE_DIR, "boites.v3i.yolov8", "data.yaml"),
    "yolo_align_train_images": os.path.join(BASE_DIR, "boites.v3i.yolov8", "train", "images"),
    "yolo_align_weights": os.path.join(BASE_DIR, "modele", "best_align.pt"),
    "aligned_dir": os.path.join(BASE_DIR, "aligned"),

    # --- Etape 3 : suppression de la partie rouge + recadrage final ---
    "final_crops_dir": os.path.join(BASE_DIR, "final_crops"),
    "final_size": (128, 384),   # (H, W) attendu par la lecture des chiffres

    # --- Etape 4 : lecture des chiffres ---
    "labels_csv": os.path.join(BASE_DIR, "essai_train", "label_filtre_officiel.csv"),
    "submission_csv": os.path.join(BASE_DIR, "submission.csv"),
}

for key in ["crops_dir", "aligned_dir", "final_crops_dir"]:
    os.makedirs(CONFIG[key], exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"BASE_DIR : {BASE_DIR}")


## 1. Détection et découpage du cadran (YOLOv8)

150 images ont été annotées manuellement avec **Roboflow** pour entraîner un premier modèle YOLOv8 (une seule classe `chiffres`) qui détecte la zone du cadran. Ce premier modèle a servi à générer 320 nouvelles images bien découpées, utilisées pour ré-entraîner un second modèle YOLO plus robuste.

Résultat rapporté : **726 images sur 795** correctement recadrées.

**Paramètres retenus :** `optimizer=AdamW`, `lr0=0.002`, `momentum=0.9`, loss = `box + cls + dfl`.


In [ ]:
def write_yolo_yaml(yaml_path, train_images_dir, val_images_dir, class_names):
    '''Génère le fichier data.yaml attendu par ultralytics.'''
    os.makedirs(os.path.dirname(yaml_path), exist_ok=True)
    data_config = {
        "train": train_images_dir,
        "val": val_images_dir,
        "nc": len(class_names),
        "names": class_names,
    }
    with open(yaml_path, "w") as f:
        yaml.dump(data_config, f, default_flow_style=False)
    print(f"YAML écrit : {yaml_path}")
    return yaml_path


In [ ]:
# --- Entraînement du modèle de détection du cadran (à lancer une seule fois) ---
write_yolo_yaml(
    CONFIG["yolo_crop_yaml"],
    CONFIG["yolo_crop_train_images"],
    CONFIG["yolo_crop_train_images"],
    class_names=["chiffres"],
)

crop_model = YOLO("yolov8n.pt")
crop_model.train(
    data=CONFIG["yolo_crop_yaml"],
    epochs=30,
    imgsz=640,
    batch=8,
    optimizer="AdamW",
    lr0=0.002,
    momentum=0.9,
    save=True,
    save_period=5,
    project=os.path.join(BASE_DIR, "YOLOv8_crop_runs"),
    name="exp_crop",
)


In [ ]:
def crop_dial_zone(model, image_folder, output_folder, conf=0.3):
    '''Pour chaque image, garde la boîte détectée avec la plus haute confiance,
    recadre l'image sur cette zone et sauvegarde le résultat.'''
    os.makedirs(output_folder, exist_ok=True)
    image_files = [f for f in os.listdir(image_folder) if f.lower().endswith((".jpg", ".jpeg", ".png"))]

    n_ok, n_missed = 0, 0
    for img_name in tqdm(image_files, desc="Découpage du cadran"):
        img_path = os.path.join(image_folder, img_name)
        img = Image.open(img_path).convert("RGB")

        results = model.predict(img_path, conf=conf, verbose=False)
        boxes = results[0].boxes

        if boxes is None or boxes.xyxy.shape[0] == 0:
            n_missed += 1
            continue

        best_idx = boxes.conf.argmax().item()
        x1, y1, x2, y2 = map(int, boxes.xyxy[best_idx])
        cropped = img.crop((x1, y1, x2, y2))
        cropped.save(os.path.join(output_folder, img_name))
        n_ok += 1

    print(f"✅ {n_ok} images recadrées, {n_missed} sans détection (sur {len(image_files)}).")


# --- Utilisation avec le modèle déjà entraîné ---
crop_model = YOLO(CONFIG["yolo_crop_weights"])
crop_dial_zone(crop_model, CONFIG["raw_images_dir"], CONFIG["crops_dir"], conf=0.3)


## 2. Alignement des cadrans

Sur les images recadrées, deux zones ont été annotées sur Roboflow : une **boîte noire** et une **boîte rouge**, correspondant aux deux extrémités du bandeau de chiffres. Un second modèle YOLOv8 (2 classes : `noir`, `rouge`) détecte ces deux repères.

À partir de leurs centres, on calcule l'angle du bandeau par rapport à l'horizontale, puis on fait pivoter l'image pour redresser le cadran — les chiffres se retrouvent alors lus de gauche à droite, dans le bon ordre, sans inclinaison.

> C'est cette boucle d'assemblage (détection → calcul d'angle → rotation → sauvegarde) qui manquait dans le notebook de travail original ; elle est reconstruite ci-dessous à partir des fonctions `compute_center`, `compute_angle` et `rotate_image` déjà présentes, plus la génération des labels YOLO (`noir`=0, `rouge`=1).


In [ ]:
write_yolo_yaml(
    CONFIG["yolo_align_yaml"],
    CONFIG["yolo_align_train_images"],
    CONFIG["yolo_align_train_images"],
    class_names=["noir", "rouge"],
)

align_model = YOLO("yolov8n.pt")
align_model.train(
    data=CONFIG["yolo_align_yaml"],
    epochs=30,
    imgsz=640,
    batch=8,
    save=True,
    save_period=5,
    project=os.path.join(BASE_DIR, "YOLOv8_align_runs"),
    name="exp_align",
)


In [ ]:
CLASS_TO_ID = {"noir": 0, "rouge": 1}


def generate_align_labels(model, image_folder, output_image_folder, output_label_folder, conf=0.4):
    '''Garde uniquement les images où noir ET rouge sont détectés, et écrit
    les labels YOLO (polygone à 4 points) pour ré-entraîner/affiner le modèle.'''
    os.makedirs(output_image_folder, exist_ok=True)
    os.makedirs(output_label_folder, exist_ok=True)

    image_files = [f for f in os.listdir(image_folder) if f.lower().endswith((".jpg", ".jpeg", ".png"))]

    for img_name in tqdm(image_files, desc="Génération des labels d'alignement"):
        img_path = os.path.join(image_folder, img_name)
        img = Image.open(img_path).convert("RGB")

        results = model.predict(img_path, conf=conf, verbose=False)
        boxes = results[0].boxes

        label_dict = {}
        for box in boxes:
            cls_id = int(box.cls[0])
            label = model.names[cls_id]
            if label in CLASS_TO_ID:
                x1, y1, x2, y2 = map(float, box.xyxy[0])
                points = [x1, y1, x2, y1, x2, y2, x1, y2, x1, y1]
                label_dict[label] = points

        if "rouge" in label_dict and "noir" in label_dict:
            shutil.copy(img_path, os.path.join(output_image_folder, img_name))
            label_path = os.path.join(output_label_folder, f"{os.path.splitext(img_name)[0]}.txt")
            with open(label_path, "w") as f:
                for label in ["noir", "rouge"]:
                    cls_id = CLASS_TO_ID[label]
                    coords = label_dict[label]
                    f.write(f"{cls_id} " + " ".join(f"{c:.2f}" for c in coords) + "\n")


In [ ]:
def compute_center(box):
    '''Centre (x, y) d'une boîte YOLO au format (x1, y1, x2, y2).'''
    x1, y1, x2, y2 = box
    return ((x1 + x2) / 2, (y1 + y2) / 2)


def compute_angle(p1, p2):
    '''Angle (en degrés) de la droite p1->p2 par rapport à l'horizontale.'''
    dx = p2[0] - p1[0]
    dy = p2[1] - p1[1]
    return math.degrees(math.atan2(dy, dx))


def rotate_image(image, angle_deg):
    '''Fait pivoter une image (array OpenCV) autour de son centre.'''
    h, w = image.shape[:2]
    center = (w // 2, h // 2)
    mat = cv2.getRotationMatrix2D(center, angle_deg, 1.0)
    rotated = cv2.warpAffine(image, mat, (w, h), flags=cv2.INTER_LINEAR,
                              borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))
    return rotated


def align_dial(model, image_folder, output_folder, conf=0.4):
    '''Boucle complète : détecte noir/rouge, calcule l'angle noir->rouge,
    redresse l'image et la sauvegarde. Les images où l'une des deux classes
    n'est pas détectée sont ignorées.'''
    os.makedirs(output_folder, exist_ok=True)
    image_files = [f for f in os.listdir(image_folder) if f.lower().endswith((".jpg", ".jpeg", ".png"))]

    n_ok, n_missed = 0, 0
    for fname in tqdm(image_files, desc="Alignement des cadrans"):
        image_path = os.path.join(image_folder, fname)
        image = cv2.imread(image_path)
        if image is None:
            n_missed += 1
            continue

        results = model.predict(image_path, conf=conf, verbose=False)
        boxes = results[0].boxes

        best_boxes = {0: None, 1: None}  # 0 = noir, 1 = rouge
        for box in boxes:
            cls_id = int(box.cls[0])
            conf_score = float(box.conf[0])
            if cls_id in best_boxes and (best_boxes[cls_id] is None or conf_score > best_boxes[cls_id][1]):
                x1, y1, x2, y2 = map(float, box.xyxy[0].tolist())
                best_boxes[cls_id] = ((x1, y1, x2, y2), conf_score)

        if best_boxes[0] is None or best_boxes[1] is None:
            n_missed += 1
            continue

        noir_center = compute_center(best_boxes[0][0])
        rouge_center = compute_center(best_boxes[1][0])
        angle = compute_angle(noir_center, rouge_center)

        rotated = rotate_image(image, angle)
        cv2.imwrite(os.path.join(output_folder, fname), rotated)
        n_ok += 1

    print(f"✅ {n_ok} images redressées, {n_missed} ignorées (repère manquant) sur {len(image_files)}.")


# --- Utilisation avec le modèle déjà entraîné ---
align_model = YOLO(CONFIG["yolo_align_weights"])
align_dial(align_model, CONFIG["crops_dir"], CONFIG["aligned_dir"], conf=0.4)


## 3. Suppression de la partie rouge & recadrage final

Une fois le cadran redressé, la partie **rouge** (les décimales, non pertinentes pour compter les m³ entiers) est neutralisée par seuillage de couleur (HSV), puis l'image est ramenée à une taille standard **128×384** par un redimensionnement "letterbox" (qui préserve les proportions et complète avec du noir plutôt que de déformer les chiffres).


In [ ]:
def remove_red_zone(image, sat_thresh=90, hue_low=(0, 10), hue_high=(170, 180)):
    '''Détecte les pixels rouges du cadran (roue des décimales) via un
    seuillage HSV et les remplace par du noir, pour ne garder que la partie
    noire (chiffres entiers, informative pour le nombre de m3).'''
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)

    mask_low = (h >= hue_low[0]) & (h <= hue_low[1])
    mask_high = (h >= hue_high[0]) & (h <= hue_high[1])
    red_mask = (mask_low | mask_high) & (s >= sat_thresh)

    result = image.copy()
    result[red_mask] = (0, 0, 0)
    return result


class LetterboxResize:
    '''Redimensionne une image en conservant son ratio, puis la complète
    (padding noir) pour atteindre exactement target_size = (H, W).'''

    def __init__(self, target_size=(128, 384), fill=0):
        self.target_h, self.target_w = target_size
        self.fill = fill

    def __call__(self, img):
        orig_w, orig_h = img.size
        scale = min(self.target_w / orig_w, self.target_h / orig_h)
        new_w, new_h = int(orig_w * scale), int(orig_h * scale)
        img_resized = img.resize((new_w, new_h))

        pad_w = self.target_w - new_w
        pad_h = self.target_h - new_h
        padded = Image.new("RGB", (self.target_w, self.target_h), (self.fill,) * 3)
        padded.paste(img_resized, (pad_w // 2, pad_h // 2))
        return padded


def build_final_dataset(image_folder, output_folder, target_size=(128, 384)):
    os.makedirs(output_folder, exist_ok=True)
    letterbox = LetterboxResize(target_size)
    image_files = [f for f in os.listdir(image_folder) if f.lower().endswith((".jpg", ".jpeg", ".png"))]

    for img_name in tqdm(image_files, desc="Suppression rouge + letterbox"):
        img_path = os.path.join(image_folder, img_name)
        cv_img = cv2.imread(img_path)
        if cv_img is None:
            continue
        cv_img = remove_red_zone(cv_img)

        pil_img = Image.fromarray(cv2.cvtColor(cv_img, cv2.COLOR_BGR2RGB))
        final_img = letterbox(pil_img)
        final_img.save(os.path.join(output_folder, img_name))


build_final_dataset(CONFIG["aligned_dir"], CONFIG["final_crops_dir"], CONFIG["final_size"])


## 4. Lecture des chiffres — solution retenue : EasyOCR

Deux architectures de CNN maison ont été testées pour prédire directement les 3 chiffres (voir l'annexe), sans succès probant : le manque de données (795 images) a conduit à un fort sur-apprentissage. C'est finalement **EasyOCR**, un modèle pré-entraîné de reconnaissance de texte (CNN + RNN + CTC), qui s'est révélé le plus efficace sur les images alignées.


In [ ]:
reader = easyocr.Reader(["en"], gpu=torch.cuda.is_available())


def read_digits(image_path, reader, n_digits=3):
    '''Lit une image et renvoie une chaîne de n_digits chiffres (tronquée /
    complétée par des zéros à gauche si besoin).'''
    img = cv2.imread(image_path)
    if img is None:
        return "ERR"

    result = reader.readtext(img, detail=0)
    raw = result[0] if result else ""
    digits_only = re.sub(r"\D", "", raw)

    digits_only = digits_only[:n_digits] if len(digits_only) > n_digits else digits_only
    return digits_only.zfill(n_digits)


def predict_folder(image_folder, reader, n_digits=3):
    image_files = sorted(f for f in os.listdir(image_folder) if f.lower().endswith((".jpg", ".jpeg", ".png")))
    rows = []
    for fname in tqdm(image_files, desc="Lecture OCR"):
        pred = read_digits(os.path.join(image_folder, fname), reader, n_digits)
        rows.append({"ID": fname, "prediction": pred})
    return pd.DataFrame(rows)


predictions_df = predict_folder(CONFIG["final_crops_dir"], reader)
predictions_df.head()


In [ ]:
# --- Evaluation, si les vrais labels sont disponibles ---
if os.path.exists(CONFIG["labels_csv"]):
    labels_df = pd.read_csv(CONFIG["labels_csv"])
    labels_df["target"] = labels_df["index_formate"].astype(str).str.zfill(3)

    merged = predictions_df.merge(labels_df[["ID", "target"]], on="ID", how="inner")
    merged["is_correct"] = merged["prediction"] == merged["target"]
    accuracy = merged["is_correct"].mean() * 100
    print(f"✅ Accuracy exacte (3 chiffres) : {accuracy:.2f}%")
    display(merged[~merged["is_correct"]].head(20))
else:
    print(f"ℹ️ Pas de fichier de labels trouvé à {CONFIG['labels_csv']} — évaluation ignorée.")


In [ ]:
# --- Génération du fichier de soumission ---
predictions_df.to_csv(CONFIG["submission_csv"], index=False)
print(f"✅ Fichier de soumission sauvegardé : {CONFIG['submission_csv']}")


## Annexe — Pistes explorées mais non retenues (CNN maison / ResNet)

Deux approches de lecture directe des 3 chiffres par un réseau entraîné from scratch ont été testées. Elles sont documentées ici pour mémoire (architectures + boucle d'entraînement reconstruite), mais **ne font pas partie du pipeline final** — c'est EasyOCR qui a été retenu (cf. section 4).

- **CNN maison** (3 blocs convolutifs + BatchNorm + ReLU + MaxPool, 3 têtes indépendantes, `CrossEntropyLoss`, dropout 30%) → **accuracy exacte : 0,68 %**.
- **ResNet18 fine-tuné** → sur-apprentissage marqué (train loss en forte baisse, validation loss plate autour de 7,1–7,3) sans généralisation, probablement dû au trop faible nombre d'images.

> La boucle d'entraînement ci-dessous est reconstruite à partir des fragments de code disponibles (elle était absente des notebooks originaux) ; elle n'a pas besoin d'être exécutée pour utiliser le pipeline EasyOCR ci-dessus.


In [ ]:
class DigitDataset(Dataset):
    '''Dataset pour la lecture directe des 3 chiffres (colonnes digit1/digit2/digit3
    attendues dans le csv de labels).'''

    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(os.path.join(self.img_dir, row["ID"])).convert("RGB")
        if self.transform:
            image = self.transform(image)
        labels = torch.tensor([row["digit1"], row["digit2"], row["digit3"]], dtype=torch.long)
        return image, labels


class SimpleMultiDigitCNN(nn.Module):
    '''CNN maison : 3 blocs conv + 3 tetes independantes (une par chiffre).'''

    def __init__(self, input_hw=(128, 384)):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
        )
        h, w = input_hw
        flat_dim = 128 * (h // 8) * (w // 8)
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(0.3)
        self.fc_shared = nn.Linear(flat_dim, 512)
        self.digit1 = nn.Linear(512, 10)
        self.digit2 = nn.Linear(512, 10)
        self.digit3 = nn.Linear(512, 10)

    def forward(self, x):
        x = self.features(x)
        x = self.flatten(x)
        x = self.dropout(F.relu(self.fc_shared(x)))
        return self.digit1(x), self.digit2(x), self.digit3(x)


class ResNetMultiDigit(nn.Module):
    '''ResNet pre-entraine, tete remplacee par 3 sorties independantes.'''

    def __init__(self, backbone="resnet18", freeze_until=9):
        super().__init__()
        base_model = getattr(models, backbone)(weights="DEFAULT")
        self.features = nn.Sequential(*list(base_model.children())[:-1])
        in_features = base_model.fc.in_features
        self.fc1 = nn.Linear(in_features, 10)
        self.fc2 = nn.Linear(in_features, 10)
        self.fc3 = nn.Linear(in_features, 10)

        for i, child in enumerate(self.features.children()):
            if i < freeze_until:
                for param in child.parameters():
                    param.requires_grad = False

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.fc1(x), self.fc2(x), self.fc3(x)


In [ ]:
def train_digit_model(model, train_loader, val_loader, device, epochs=20, lr=1e-3):
    '''Boucle d'entrainement generique (reconstruite) pour un modele a 3 tetes.
    Retourne le modele entraine et l'historique des pertes.'''
    model = model.to(device)
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    criterion = nn.CrossEntropyLoss()

    train_losses, val_losses = [], []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [train]", leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            out1, out2, out3 = model(imgs)
            loss = (criterion(out1, labels[:, 0]) + criterion(out2, labels[:, 1]) + criterion(out3, labels[:, 2])) / 3
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)
        train_loss = running_loss / len(train_loader.dataset)
        train_losses.append(train_loss)

        model.eval()
        running_val_loss = 0.0
        with torch.no_grad():
            for imgs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [val]", leave=False):
                imgs, labels = imgs.to(device), labels.to(device)
                out1, out2, out3 = model(imgs)
                loss = (criterion(out1, labels[:, 0]) + criterion(out2, labels[:, 1]) + criterion(out3, labels[:, 2])) / 3
                running_val_loss += loss.item() * imgs.size(0)
        val_loss = running_val_loss / len(val_loader.dataset)
        val_losses.append(val_loss)

        print(f"Epoch {epoch+1}/{epochs} — train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")

    return model, train_losses, val_losses


def evaluate_exact_accuracy(model, loader, device):
    '''Accuracy exacte : les 3 chiffres doivent etre corrects simultanement.'''
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc="Evaluation"):
            imgs = imgs.to(device)
            out1, out2, out3 = model(imgs)
            preds = torch.stack([out1.argmax(1), out2.argmax(1), out3.argmax(1)], dim=1).cpu()
            correct += (preds == labels).all(dim=1).sum().item()
            total += labels.size(0)
    accuracy = correct / total
    print(f"✅ Accuracy exacte (3 chiffres corrects) : {accuracy:.4f}")
    return accuracy


# --- Exemple d'utilisation (nécessite le csv de labels avec digit1/digit2/digit3) ---
# df = pd.read_csv(CONFIG["labels_csv"])
# transform = transforms.Compose([
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.5] * 3, std=[0.5] * 3),
# ])
# dataset = DigitDataset(df, CONFIG["final_crops_dir"], transform=transform)
# n_val = int(0.2 * len(dataset))
# train_ds, val_ds = random_split(dataset, [len(dataset) - n_val, n_val])
# train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
# val_loader = DataLoader(val_ds, batch_size=32)
#
# model = SimpleMultiDigitCNN(input_hw=CONFIG["final_size"])
# model, train_losses, val_losses = train_digit_model(model, train_loader, val_loader, DEVICE, epochs=20)
# evaluate_exact_accuracy(model, val_loader, DEVICE)
#
# plt.plot(train_losses, label="Train Loss")
# plt.plot(val_losses, label="Validation Loss")
# plt.title("Courbes de perte")
# plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.grid(True); plt.legend(); plt.show()


## Conclusion

Le pipeline retenu enchaîne **YOLOv8** (détection puis alignement du cadran), un **post-traitement** (suppression de la zone rouge, letterbox 128×384) et **EasyOCR** pour la lecture finale des 3 chiffres, avec un score de soumission d'environ **0,284**.

**Difficultés rencontrées :**
- Faible quantité de données (795 images seulement) pour une tâche de vision par ordinateur.
- Variations d'angle, de luminosité et de netteté des photos.
- Généralisation difficile de la rotation automatique sur les cas les plus dégradés.

**Pistes d'amélioration (identifiées dans le rapport) :**
- Découper chaque chiffre individuellement plutôt que de lire le bandeau entier.
- Améliorer le prétraitement (rehaussement du contraste, suppression du bruit).
- Enrichir le dataset avec des augmentations artificielles.
- Fine-tuner le modèle OCR directement sur des chiffres de compteurs.
